## Create a simple optimization model using Juila code

In [4]:
using Pkg
Pkg.add("JuMP")
Pkg.add("CPLEX") #solver

     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [5]:
using JuMP
using CPLEX

### Super Simple

In [6]:
mymodel = Model(CPLEX.Optimizer) #create a model called "mymodel" and use the HiGHS solver

@variable(mymodel, x >= 0) #create a decision variable called "x" that is greater than or equal to 0
@variable(mymodel, y >= 0,Int) #create a decision variable called "y" that is greater than or equal to 0

@objective(mymodel, Max, 30.5x + 20.2y) #create an objective function that maximizes x + y

@constraint(mymodel, x+2y <=100) #create a constraint that 2x + y is less than or equal to 100
@constraint(mymodel, x<=100) #create a constraint that x is less than or equal to 4



x ≤ 100

In [7]:
optimize!(mymodel) #optimize the model

println("x = ", value(x)) #print the value of x
println("y = ", value(y)) #print the value of y
println("Profit = ", objective_value(mymodel)) #print the value of the objective function

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 2 rows and 2 columns.
All rows and columns eliminated.
Presolve time = 0.00 sec. (0.00 ticks)
x = 100.0
y = 0.0
Profit = 3050.0

Root node processing (before b&c):
  Real time             =    0.01 sec. (0.00 ticks)
Parallel b&c, 8 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.01 sec. (0.00 ticks)


## Water Balance Optimization

### Create Model

In [54]:
water_balance_model = Model(CPLEX.Optimizer)

A JuMP Model
├ solver: CPLEX
├ objective_sense: FEASIBILITY_SENSE
├ num_variables: 0
├ num_constraints: 0
└ Names registered in the model: none

### Create Indices and Constants

In [55]:
facilities = ["A","B"]
months = [1,2,3]
months0 = [0,1,2,3]
commodities = ["irrigation","power"]

f1_mwhtokaf = 200
f2_mwhtokaf = 150

150

### Create Parameters

In [56]:
#revenue per unit of commodities for each facility
revenue = Dict(("irrigation","A")=>43000, ("irrigation","B")=>43000,("power","A")=>25*f1_mwhtokaf, ("power","B")=>25*f2_mwhtokaf)

#starting value for each facility
start = Dict("A"=>1300,"B"=>1000)

#max reservoir level
max_water = Dict(("A",1)=>1500,("A",2)=>1500,("A",3)=>1500,("B",1)=>1100,("B",2)=>1100,("B",3)=>1100)

#min reservoir level
min_water = Dict(("A",1)=>1200,("A",2)=>1200,("A",3)=>1200,("B",1)=>900,("B",2)=>900,("B",3)=>900)

#estimated inflow
inflow = Dict(("A",1)=>300,("A",2)=>260,("A",3)=>400,("B",1)=>100,("B",2)=>175,("B",3)=>190)

#maximum sold for irrigation
max_irrigation = Dict(("A",1)=>150,("A",2)=>150,("A",3)=>75,("B",1)=>100,("B",2)=>100,("B",3)=>50)

#water for maximum power
max_power = Dict(("A",1)=>324,("A",2)=>324,("A",3)=>324,("B",1)=>240,("B",2)=>240,("B",3)=>240)

#minimum power from A
min_power_A = Dict(("A",1)=>234,("A",2)=>234,("A",3)=>0)

#minimum water out of B
min_out_B = Dict(("B",1)=>200,("B",2)=>200,("B",3)=>200)


Dict{Tuple{String, Int64}, Int64} with 3 entries:
  ("B", 1) => 200
  ("B", 2) => 200
  ("B", 3) => 200

### Create Decision Variables

In [57]:
#powergen
@variable(water_balance_model, powergen[facilities,months]>=0)

#watersold
@variable(water_balance_model, watersold[facilities,months]>=0)

#spill
@variable(water_balance_model,spill[facilities,months]>=0)

#remainingwater
@variable(water_balance_model,remainingwater[facilities,months0]>=0)



2-dimensional DenseAxisArray{VariableRef,2,...} with index sets:
    Dimension 1, ["A", "B"]
    Dimension 2, [0, 1, 2, 3]
And data, a 2×4 Matrix{VariableRef}:
 remainingwater[A,0]  remainingwater[A,1]  …  remainingwater[A,3]
 remainingwater[B,0]  remainingwater[B,1]     remainingwater[B,3]

In [58]:
#set the initial water level at the end of month 0
for f in facilities
    fix(remainingwater[f, 0], start[f]; force=true)
end

### Create Objective Function

In [59]:
@objective(water_balance_model,Max, 
    sum(
        powergen[f,m]*revenue[("power",f)]+
        watersold[f,m]*revenue[("irrigation",f)]
            for f in facilities for m in months )
)

5000 powergen[A,1] + 43000 watersold[A,1] + 5000 powergen[A,2] + 43000 watersold[A,2] + 5000 powergen[A,3] + 43000 watersold[A,3] + 3750 powergen[B,1] + 43000 watersold[B,1] + 3750 powergen[B,2] + 43000 watersold[B,2] + 3750 powergen[B,3] + 43000 watersold[B,3]

### Define Constraints

In [60]:
#min reservoir
@constraint(water_balance_model,[f in facilities,m in months],remainingwater[f,m]>=min_water[f,m])

#max reservoir
@constraint(water_balance_model,[f in facilities,m in months],remainingwater[f,m]<=max_water[f,m])

#max irrigation
@constraint(water_balance_model,[f in facilities,m in months],watersold[f,m]<=max_irrigation[f,m])

#water balance
#cannot do an if statement directly in the @constraint nor define a function outside and use that
for f in facilities, m in months
    if f == "A"
        @constraint(
            water_balance_model,
            remainingwater[f, m] ==
                remainingwater[f, m - 1] +
                inflow[f, m] -
                powergen[f, m] -
                watersold[f, m] -
                spill[f, m]
        )
    else
        @constraint(
            water_balance_model,
            remainingwater[f, m] ==
                remainingwater[f, m - 1] +
                powergen["A", m] +
                spill["A", m] +
                inflow[f, m] -
                powergen[f, m] -
                watersold[f, m] -
                spill[f, m]
        )
    end
end

#max power gen
@constraint(water_balance_model,[f in facilities,m in months],powergen[f,m]<=max_power[f,m])

#min power gen for a
@constraint(water_balance_model,[m in months],powergen["A",m]>=min_power_A["A",m])

#min water from B
@constraint(water_balance_model,[m in months],sum(powergen["B",m]+spill["B",m])>=min_out_B["B",m])




1-dimensional DenseAxisArray{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape},1,...} with index sets:
    Dimension 1, [1, 2, 3]
And data, a 3-element Vector{ConstraintRef{Model, MathOptInterface.ConstraintIndex{MathOptInterface.ScalarAffineFunction{Float64}, MathOptInterface.GreaterThan{Float64}}, ScalarShape}}:
 powergen[B,1] + spill[B,1] ≥ 200
 powergen[B,2] + spill[B,2] ≥ 200
 powergen[B,3] + spill[B,3] ≥ 200

### Solve

In [64]:
optimize!(water_balance_model) #optimize the model

for f in facilities 
    for m in months
        println("power, facility= ",f," month= ",m, " :",value(powergen[f,m]))
    end
end

println("Total profit: ",objective_value(water_balance_model))

CPLEX Error  3003: Not a mixed-integer problem.
Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
Using devex.
power, facility= A month= 1 :234.0
power, facility= A month= 2 :234.0
power, facility= A month= 3 :324.0
power, facility= B month= 1 :240.0
power, facility= B month= 2 :240.0
power, facility= B month= 3 :240.0
Total profit: 2.8891e7
